# Historical Relation Extraction Assignment

## Overview
This notebook demonstrates the extraction of historical relations (`at` and `isAt`) between Persons and Places using a fine-tuned LLM (Llama-3.1-8B) with adapter switching (PEFT).

### Structure
1. **Environment Setup**: Installing dependencies like `unsloth`.
2. **Data & Utilities**: Defining robust JSON harvesting and data loading functions.
3. **Prompting Strategy**: Setting up the few-shot or zero-shot prompts for relation extraction.
4. **Inference**: Running inference with adapter switching between the `at` and `isAt` tasks.
5. **Evaluation**: Comparing model predictions against gold labels.

### 1. Environment & Setup
Unsloth 'Nuclear' installation and Google Drive mounting.

In [ ]:
# Unsloth "Nuclear" installation script
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-dpyko5wt/unsloth_cc59ae7124ca47a196127a01a913fd27
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-dpyko5wt/unsloth_cc59ae7124ca47a196127a01a913fd27
  Resolved https://github.com/unslothai/unsloth.git to commit 220ff5aabaa67ede8a29af0859297c7ecac96985
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 138.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 26.8 MB/s eta 0:00:00
  

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import re

# Base Paths
BASE_PATH = "/content/drive/MyDrive/colab_data/HIPE-2026-data"
AT_ADAPTER = os.path.join(BASE_PATH, "trained_models/llama_8b_at_adapter_synthetic_reasoning_early_stopping")
ISAT_ADAPTER = os.path.join(BASE_PATH, "trained_models/llama_8b_isAt_adapter_synthetic_reasoning_early_stopping")
DATA_PATH = os.path.join(BASE_PATH, "data/sandbox")

LANGUAGES = ['en', 'de', 'fr']

### 2. Specialized Logic Requirements
Robust harvester and data loading utilities.

In [ ]:
def harvest_json_robust(text):
    # Normalize smart quotes
    text = text.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")

    try:
        # Attempt to parse the entire text as a single JSON object
        full_json = json.loads(text)
        # If it's a dictionary with a 'results' key that's a non-empty list
        if isinstance(full_json, dict) and "results" in full_json and \
           isinstance(full_json["results"], list) and len(full_json["results"]) > 0:
            # Return the first item from the 'results' list, wrapped in a list
            # This is to make it compatible with the existing iteration logic `for p in parsed:`
            return [full_json["results"][0]]
        # If it's a dictionary but doesn't have the 'results' structure, return it as is (wrapped in a list)
        elif isinstance(full_json, dict):
            return [full_json]
    except json.JSONDecodeError:
        # If direct parsing fails, proceed to regex-based extraction
        pass

    # Fallback: Extract JSON objects using regex (original logic, for simple cases or partial outputs)
    matches = re.findall(r'\{[^{}]*\}', text)
    results = []
    for match in matches:
        # Correct unquoted labels
        match = re.sub(r':\s*TRUE\b', ': "TRUE"', match, flags=re.IGNORECASE)
        match = re.sub(r':\s*FALSE\b', ': "FALSE"', match, flags=re.IGNORECASE)
        match = re.sub(r':\s*PROBABLE\b', ': "PROBABLE"', match, flags=re.IGNORECASE)
        try:
            results.append(json.loads(match))
        except json.JSONDecodeError:
            pass

    # Return results if any were found, otherwise an empty list to indicate no valid JSON was harvested
    return results if results else []

def load_data(lang):
    filepath = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")
    data = []
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
    else:
        print(f"Warning: {filepath} not found.")
    return data

### 3. Prompt Templates
Defining the prompt templates matching the training setup.

In [ ]:
at_prompt = """You are an expert computational historian specializing in relation extraction.
TASK: Determine the historical relation 'at' between the designated Persons and Places based on the text.

CRITICAL LOGICAL CONSTRAINTS:
1. 'at' represents a permanent, structural, institutional, professional, or residency-based geographic connection over time.
2. Use 'TRUE' ONLY if there is 100% certainty and explicit absolute proof of the connection.
3. Use 'PROBABLE' for 'at' when strong contextual, regional, or family/professional affiliation implies geographic connectivity without explicit absolute proof.
4. Use 'FALSE' if no evidence is present or the context contradicts such a relation.

TARGET TEXT FOR ANALYSIS:
"{text}"

### OUTPUT INSTRUCTIONS ###
- Output ONLY the JSON object. START your response with '{{'.
- The JSON must contain a "reasoning" key for your chain-of-thought, followed by a "results" array.
- The format must be exactly:
{{
  "reasoning": "your step-by-step logic here",
  "results": [{{"at": "VALUE"}}]
}}
where VALUE is TRUE, FALSE, or PROBABLE.

PAIRS TO EVALUATE:
{pairs_list_str}"""

isAt_prompt = """You are an expert computational historian specializing in relation extraction.
TASK: Determine the temporal relation 'isAt' between the designated Persons and Places based on the text.

CRITICAL LOGICAL CONSTRAINTS:
1. 'isAt' represents literal immediate physical presence at that place within the narrative moment (the temporal horizon of the article).
2. 'isAt' is TRUE if there is evidence the person was at the location up to about one month before the publication date.
3. Use 'FALSE' if the person is elsewhere, the event happened in the distant past, or no evidence of current presence exists.

TARGET TEXT FOR ANALYSIS:
"{text}"

### OUTPUT INSTRUCTIONS ###
- Output ONLY the JSON object. START your response with '{{'.
- The JSON must contain a "reasoning" key for your chain-of-thought, followed by a "results" array.
- The format must be exactly:
{{
  "reasoning": "your step-by-step logic here",
  "results": [{{"isAt": "VALUE"}}]
}}
where VALUE is TRUE or FALSE.

PAIRS TO EVALUATE:
{pairs_list_str}"""

def format_chat_prompt(relation, person, place, text):
    pairs_list_str = f"Person: {person}, Place: {place}"
    if relation == 'at':
        user_msg = at_prompt.format(text=text, pairs_list_str=pairs_list_str)
    else:
        user_msg = isAt_prompt.format(text=text, pairs_list_str=pairs_list_str)

    return [{"role": "user", "content": user_msg}]


### 4. Sequential Inference
Loading base model, `at` inference, followed by adapter switch to `isAt` and Logic Guard.

In [ ]:
from unsloth import FastLanguageModel
import torch
import sys

max_seq_length = 4096

if not os.path.exists(AT_ADAPTER):
    print(f"Error: AT adapter path does not exist: {AT_ADAPTER}")
    sys.exit(1)

# Load base model
print("Loading Base Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Base Model...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
from tqdm.notebook import tqdm

# 1. AT Inference
print(f"Loading AT-Specialist from {AT_ADAPTER}...")
if "at_adapter" not in getattr(model, "peft_config", {}):
    model.load_adapter(AT_ADAPTER, adapter_name="at_adapter")
model.set_adapter("at_adapter")
FastLanguageModel.for_inference(model)

run_lang = "en"

if 'at_predictions' not in locals():
    at_predictions = {}
at_predictions[run_lang] = []

print(f"Running AT inference for {run_lang.upper()}...")
data = load_data(run_lang)

for item in tqdm(data, desc="Processing Documents (AT)"):
    for pair in item.get('sampled_pairs', []):
        pers_list = pair.get('pers_mentions_list', [])
        loc_list = pair.get('loc_mentions_list', [])
        person = pers_list[0] if pers_list else ""
        place = loc_list[0] if loc_list else ""

        messages = format_chat_prompt('at', person, place, item.get('text', ''))
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

        max_retries = 3
        for attempt in range(max_retries):
            # Give it enough tokens for reasoning and remove stop_strings
            outputs = model.generate(input_ids=inputs, max_new_tokens=2048, do_sample=False, tokenizer=tokenizer)
            response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

            resp_clean = response.strip().upper()
            if resp_clean in ['TRUE', 'FALSE', 'PROBABLE']:
                pred = resp_clean
            else:
                parsed = harvest_json_robust(response)
                pred = 'ERROR'
                for p in parsed:
                    if isinstance(p, dict):
                        for val in p.values():
                            if str(val).upper() in ['TRUE', 'FALSE', 'PROBABLE']:
                                pred = str(val).upper()
                                break
                    if pred != 'ERROR':
                        break

                if pred == 'ERROR':
                    if 'PROBABLE' in resp_clean:
                        pred = 'PROBABLE'
                    elif 'TRUE' in resp_clean:
                        pred = 'TRUE'
                    elif 'FALSE' in resp_clean:
                        pred = 'FALSE'

            if pred != 'ERROR':
                break
            elif attempt < max_retries - 1:
                print(f"  [AT Retry {attempt+1}] Model returned error, re-prompting...")

        if pred == 'ERROR':
            print(f"[AT ERROR] doc: {item.get('document_id')} | pair: {person}-{place} | Raw Response: {response}")
            pred = 'FALSE' # Safety net to prevent schema failure

        pair['at'] = pred

    at_predictions[run_lang].append(item)


Loading AT-Specialist from /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_8b_at_adapter_synthetic_reasoning_early_stopping...


Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

Running AT inference for EN...


Processing Documents (AT):   0%|          | 0/17 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_att

In [ ]:
import torch
import gc

# Delete the model and trainer from memory
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Now load the base model fresh
print("Loading Base Model for ISAT...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

# Load the isAt adapter
print(f"Loading ISAT-Specialist from {ISAT_ADAPTER}...")
model.load_adapter(ISAT_ADAPTER, adapter_name="isAt_adapter")
model.set_adapter("isAt_adapter")
FastLanguageModel.for_inference(model)

print("Ready to run ISAT inference loop.")


Loading Base Model for ISAT...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


Loading ISAT-Specialist from /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_8b_isAt_adapter_synthetic_reasoning_early_stopping...


Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

Ready to run ISAT inference loop.


In [ ]:
from tqdm.notebook import tqdm
import time
import json
import os

# 2. ISAT Inference
run_lang = "en"

if 'final_results' not in locals():
    final_results = {}
final_results[run_lang] = []

print(f"Running ISAT inference for {run_lang.upper()}...")

if run_lang not in at_predictions or not at_predictions[run_lang]:
    print(f"Error: You must run the AT Inference block for '{run_lang}' first!")
else:
    total_docs = len(at_predictions[run_lang])
    for doc_idx, item in enumerate(tqdm(at_predictions[run_lang], desc="Processing Documents (ISAT)")):
        doc_id = item.get('document_id', 'Unknown')
        pairs = item.get('sampled_pairs', [])


        for i, pair in enumerate(pairs):
            pers_list = pair.get('pers_mentions_list', [])
            loc_list = pair.get('loc_mentions_list', [])
            person = pers_list[0] if pers_list else ""
            place = loc_list[0] if loc_list else ""

            messages = format_chat_prompt('isAt', person, place, item.get('text', ''))
            inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

            max_retries = 3
            for attempt in range(max_retries):
                start_time = time.time()
                # Give it enough tokens for reasoning and remove stop_strings
                outputs = model.generate(input_ids=inputs, max_new_tokens=2048, do_sample=False, use_cache=True, tokenizer=tokenizer)
                end_time = time.time()

                gen_duration = end_time - start_time
                response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

                resp_clean = response.strip().upper()
                if resp_clean in ['TRUE', 'FALSE']:
                    isAt_pred = resp_clean
                else:
                    parsed = harvest_json_robust(response)
                    isAt_pred = 'ERROR'
                    for p in parsed:
                        if isinstance(p, dict):
                            for val in p.values():
                                if str(val).upper() in ['TRUE', 'FALSE']:
                                    isAt_pred = str(val).upper()
                                    break
                        if isAt_pred != 'ERROR':
                            break

                if isAt_pred == 'ERROR':
                    if 'TRUE' in resp_clean:
                        isAt_pred = 'TRUE'
                    elif 'FALSE' in resp_clean:
                        isAt_pred = 'FALSE'

                if isAt_pred != 'ERROR':
                    break
                elif attempt < max_retries - 1:
                    print(f"    [ISAT Retry {attempt+1}] Model returned error, re-prompting...")

            if isAt_pred == 'ERROR':
                print(f"[isAt ERROR] doc: {doc_id} | pair: {person}-{place} | Raw Response: {response}")
                isAt_pred = 'FALSE' # Safety net to prevent schema failure

            pair['isAt'] = isAt_pred

        final_results[run_lang].append(item)

    out_path = os.path.join(BASE_PATH, f"integrated_llama_8b_{run_lang}_synthetic_reasoning_results.jsonl")
    with open(out_path, 'w', encoding='utf-8') as f:
        for res in final_results[run_lang]:
            f.write(json.dumps(res) + '\n')
    print(f"\nDONE. Saved final predictions to {out_path}")


Running ISAT inference for EN...


Processing Documents (ISAT):   0%|          | 0/17 [00:00<?, ?it/s]

Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/t


DONE. Saved final predictions to /content/drive/MyDrive/colab_data/HIPE-2026-data/integrated_llama_8b_en_synthetic_reasoning_results.jsonl


### 5. Evaluation
Automatically trigger the official scorer scripts for each integrated output file.

In [ ]:
import json
import os

print("Running automated evaluation script...")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_llama_8b_{lang}_synthetic_reasoning_results.jsonl")
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if os.path.exists(pred_path) and os.path.exists(gold_path):
        print(f"\nEvaluating {lang}...")
        # cd into BASE_PATH so the script can find the 'schemas/' directory
        !cd "{BASE_PATH}" && python scripts/file_scorer_evaluation.py --predictions_file "{pred_path}" --gold_data_file "{gold_path}"
    else:
        print(f"Skipping eval for {lang}. Check if prediction or gold files exist.")

Running automated evaluation script...

Evaluating en...

Evaluation Results for integrated_llama_8b_en_synthetic_reasoning_results.jsonl:
  'at': macro_recall=0.4533, accuracy=0.4768 (72/151)
  'isAt': macro_recall=0.6955, accuracy=0.4636 (70/151)
  'global': macro_recall=0.5744 (142/302)


Evaluating de...

Evaluation Results for integrated_llama_8b_de_synthetic_reasoning_results.jsonl:
  'at': macro_recall=0.4382, accuracy=0.4653 (201/432)
  'isAt': macro_recall=0.7427, accuracy=0.6991 (302/432)
  'global': macro_recall=0.5905 (503/864)

Skipping eval for fr. Check if prediction or gold files exist.


In [ ]:
import json
import os

print("Detailed Prediction Analysis by Label")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_llama_8b_{lang}_synthetic_reasoning_results.jsonl")
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if not (os.path.exists(pred_path) and os.path.exists(gold_path)):
        print(f"Missing files for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Analysis for {lang.upper()} ---")
    print(f"{'='*40}")

    # 1. Load gold data into a dictionary mapped by document_id
    gold_data = {}
    with open(gold_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            doc_id = item['document_id']
            # Map pair entities to their gold pairs so order doesn't matter
            gold_data[doc_id] = {}
            for pair in item.get('sampled_pairs', []):
                pers_id = pair.get('pers_entity_id')
                loc_id = pair.get('loc_entity_id')
                gold_data[doc_id][(pers_id, loc_id)] = pair

    # 2. Track stats
    at_stats = {
        "TRUE": {"correct": 0, "wrong": 0},
        "FALSE": {"correct": 0, "wrong": 0},
        "PROBABLE": {"correct": 0, "wrong": 0}
    }
    isat_stats = {
        "TRUE": {"correct": 0, "wrong": 0},
        "FALSE": {"correct": 0, "wrong": 0}
    }

    # 3. Compare predictions against gold
    with open(pred_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            doc_id = item['document_id']
            pred_pairs = item.get('sampled_pairs', [])

            for p_pair in pred_pairs:
                pers_id = p_pair.get('pers_entity_id')
                loc_id = p_pair.get('loc_entity_id')

                # Find corresponding gold pair
                g_pair = gold_data.get(doc_id, {}).get((pers_id, loc_id))

                if not g_pair:
                    continue # Skip if no matching gold pair found

                # --- Check 'at' Field ---
                g_at = g_pair.get('at', 'FALSE')
                p_at = p_pair.get('at', 'ERROR')

                if g_at in at_stats:
                    if g_at == p_at:
                        at_stats[g_at]['correct'] += 1
                    else:
                        at_stats[g_at]['wrong'] += 1

                # --- Check 'isAt' Field ---
                g_isat = g_pair.get('isAt', 'FALSE')
                p_isat = p_pair.get('isAt', 'ERROR')

                if g_isat in isat_stats:
                    if g_isat == p_isat:
                        isat_stats[g_isat]['correct'] += 1
                    else:
                        isat_stats[g_isat]['wrong'] += 1

    # 4. Print Results
    print("\n[ 'AT' FIELD STATS ]")
    for label, counts in at_stats.items():
        total = counts['correct'] + counts['wrong']
        acc = (counts['correct'] / total * 100) if total > 0 else 0
        print(f"  Gold={label:<8}: {counts['correct']:>4} Correct | {counts['wrong']:>4} Wrong | Recall: {acc:>5.1f}%")

    print("\n[ 'ISAT' FIELD STATS ]")
    for label, counts in isat_stats.items():
        total = counts['correct'] + counts['wrong']
        acc = (counts['correct'] / total * 100) if total > 0 else 0
        print(f"  Gold={label:<8}: {counts['correct']:>4} Correct | {counts['wrong']:>4} Wrong | Recall: {acc:>5.1f}%")


Detailed Prediction Analysis by Label

--- Analysis for EN ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :    5 Correct |   24 Wrong | Accuracy:  17.2%
  Gold=FALSE   :   10 Correct |   58 Wrong | Accuracy:  14.7%
  Gold=PROBABLE:   54 Correct |    0 Wrong | Accuracy: 100.0%

[ 'ISAT' FIELD STATS ]
  Gold=TRUE    :   18 Correct |    0 Wrong | Accuracy: 100.0%
  Gold=FALSE   :   67 Correct |   66 Wrong | Accuracy:  50.4%

--- Analysis for DE ---

[ 'AT' FIELD STATS ]
  Gold=TRUE    :    5 Correct |   36 Wrong | Accuracy:  12.2%
  Gold=FALSE   :   52 Correct |  192 Wrong | Accuracy:  21.3%
  Gold=PROBABLE:  144 Correct |    3 Wrong | Accuracy:  98.0%

[ 'ISAT' FIELD STATS ]
  Gold=TRUE    :   23 Correct |    6 Wrong | Accuracy:  79.3%
  Gold=FALSE   :  279 Correct |  124 Wrong | Accuracy:  69.2%
Missing files for fr


In [ ]:
import json
import os

print("Prediction Distribution (Model Guesses)")

for lang in LANGUAGES:
    pred_path = os.path.join(BASE_PATH, f"integrated_llama_8b_{lang}_synthetic_reasoning_results.jsonl")

    if not os.path.exists(pred_path):
        print(f"Missing prediction file for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Model Guesses for {lang.upper()} ---")
    print(f"{'='*40}")

    at_guesses = {"TRUE": 0, "FALSE": 0, "PROBABLE": 0, "ERROR": 0}
    isat_guesses = {"TRUE": 0, "FALSE": 0, "ERROR": 0}

    with open(pred_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            for pair in item.get('sampled_pairs', []):
                p_at = pair.get('at', 'ERROR')
                p_isat = pair.get('isAt', 'ERROR')

                if p_at in at_guesses:
                    at_guesses[p_at] += 1
                else:
                    at_guesses['ERROR'] += 1

                if p_isat in isat_guesses:
                    isat_guesses[p_isat] += 1
                else:
                    isat_guesses['ERROR'] += 1

    print("\n[ 'AT' FIELD GUESSES ]")
    for label, count in at_guesses.items():
        if count > 0 or label in ["TRUE", "FALSE", "PROBABLE"]:
            print(f"  Guessed {label:<8}: {count:>4} times")

    print("\n[ 'ISAT' FIELD GUESSES ]")
    for label, count in isat_guesses.items():
        if count > 0 or label in ["TRUE", "FALSE"]:
            print(f"  Guessed {label:<8}: {count:>4} times")

Prediction Distribution (Model Guesses)

--- Model Guesses for EN ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :    5 times
  Guessed FALSE   :   10 times
  Guessed PROBABLE:  136 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :   84 times
  Guessed FALSE   :   67 times

--- Model Guesses for DE ---

[ 'AT' FIELD GUESSES ]
  Guessed TRUE    :    9 times
  Guessed FALSE   :   52 times
  Guessed PROBABLE:  371 times

[ 'ISAT' FIELD GUESSES ]
  Guessed TRUE    :  147 times
  Guessed FALSE   :  285 times
Missing prediction file for fr


In [ ]:
import json
import os

print("Gold Label Distribution (Actual Data)")

for lang in LANGUAGES:
    gold_path = os.path.join(DATA_PATH, f"{lang}-dev-cleaned.jsonl")

    if not os.path.exists(gold_path):
        print(f"Missing gold file for {lang}")
        continue

    print(f"\n{'='*40}")
    print(f"--- Gold Labels for {lang.upper()} ---")
    print(f"{'='*40}")

    at_counts = {"TRUE": 0, "FALSE": 0, "PROBABLE": 0, "ERROR": 0}
    isat_counts = {"TRUE": 0, "FALSE": 0, "ERROR": 0}

    with open(gold_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            for pair in item.get('sampled_pairs', []):
                g_at = pair.get('at', 'ERROR')
                g_isat = pair.get('isAt', 'ERROR')

                if g_at in at_counts:
                    at_counts[g_at] += 1
                else:
                    at_counts['ERROR'] += 1

                if g_isat in isat_counts:
                    isat_counts[g_isat] += 1
                else:
                    isat_counts['ERROR'] += 1

    print("\n[ 'AT' FIELD GOLD LABELS ]")
    for label, count in at_counts.items():
        if count > 0 or label in ["TRUE", "FALSE", "PROBABLE"]:
            print(f"  Actual {label:<8}: {count:>4} times")

    print("\n[ 'ISAT' FIELD GOLD LABELS ]")
    for label, count in isat_counts.items():
        if count > 0 or label in ["TRUE", "FALSE"]:
            print(f"  Actual {label:<8}: {count:>4} times")

Gold Label Distribution (Actual Data)

--- Gold Labels for EN ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :   29 times
  Actual FALSE   :   68 times
  Actual PROBABLE:   54 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :   18 times
  Actual FALSE   :  133 times

--- Gold Labels for DE ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :   41 times
  Actual FALSE   :  244 times
  Actual PROBABLE:  147 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :   29 times
  Actual FALSE   :  403 times

--- Gold Labels for FR ---

[ 'AT' FIELD GOLD LABELS ]
  Actual TRUE    :  179 times
  Actual FALSE   :  952 times
  Actual PROBABLE:  367 times

[ 'ISAT' FIELD GOLD LABELS ]
  Actual TRUE    :  127 times
  Actual FALSE   : 1371 times


Debugging/testing one run to ensure json is output correctly

In [ ]:
import json

print("=== DEBUGGING INFERENCE (FIRST 10 PAIRS) ===\n")

test_lang = 'de'
print(f"Loading data for: {test_lang.upper()}")
test_data = load_data(test_lang)

if not test_data:
    print(f"Could not load {test_lang.upper()} data!")
else:
    print("\n--- Switching to AT Adapter ---")
    if "at_adapter" not in getattr(model, "peft_config", {}):
        model.load_adapter(AT_ADAPTER, adapter_name="at_adapter")
    model.set_adapter("at_adapter")
    FastLanguageModel.for_inference(model)

    pair_count = 0
    max_pairs = 10

    for item in test_data:
        if pair_count >= max_pairs: break
        text = item.get('text', '')

        for pair in item.get('sampled_pairs', []):
            if pair_count >= max_pairs: break

            pers_list = pair.get('pers_mentions_list', [])
            loc_list = pair.get('loc_mentions_list', [])
            person = pers_list[0] if pers_list else ""
            place = loc_list[0] if loc_list else ""
            gold_at = pair.get('at', 'UNKNOWN')

            print(f"\n{'='*50}")
            print(f"[{pair_count+1}] Testing Person: '{person}' | Place: '{place}'")
            print(f"Gold 'at' label: {gold_at}")
            print(f"{'='*50}")

            messages = format_chat_prompt('at', person, place, text)
            inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

            # Allow up to 512 tokens to read the reasoning
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=512,
                do_sample=False,
                tokenizer=tokenizer
            )
            response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

            print("\n--- RAW MODEL OUTPUT ---")
            print(response)
            print("------------------------\n")

            # Test extraction logic
            parsed = harvest_json_robust(response)
            pred = 'ERROR'
            resp_clean = response.strip().upper()

            if resp_clean in ['TRUE', 'FALSE', 'PROBABLE']:
                pred = resp_clean
            else:
                for p in parsed:
                    if isinstance(p, dict):
                        for val in p.values():
                            if str(val).upper() in ['TRUE', 'FALSE', 'PROBABLE']:
                                pred = str(val).upper()
                                break
                    if pred != 'ERROR': break

                # Fallback
                if pred == 'ERROR':
                    if 'PROBABLE' in resp_clean: pred = 'PROBABLE'
                    elif 'TRUE' in resp_clean: pred = 'TRUE'
                    elif 'FALSE' in resp_clean: pred = 'FALSE'

            print(f"Parsed JSON: {parsed}")
            print(f"Extracted Prediction: {pred}")

            if pred == gold_at:
                print("✅ Match!")
            else:
                print("❌ Mismatch!")

            pair_count += 1


=== DEBUGGING INFERENCE (FIRST 10 PAIRS) ===

Loading data for: DE


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Switching to AT Adapter ---

[1] Testing Person: 'Anna Lerch' | Place: 'Lusern'
Gold 'at' label: FALSE


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- RAW MODEL OUTPUT ---
{
  "reasoning": "The relationship between Anna Lerch and Lusern is false because the text only mentions Anna Lerch in the context of her company's bankruptcy, without providing any information about her connection to Lusern. There is no mention of Anna Lerch being from or associated with Lusern, making the relationship false.",
  "results": [{"at": "FALSE"}]}

------------------------

Parsed JSON: [{'at': 'FALSE'}]
Extracted Prediction: FALSE
✅ Match!

[2] Testing Person: 'Alkred Sehraner, Kaukmann' | Place: 'Guls'
Gold 'at' label: PROBABLE


...


--- RAW MODEL OUTPUT ---
{
  "reasoning": "The relationship between Bruno Wild, Ingenieur, and Olten is probable because the text mentions \"Volksmagazin. Olten\" in the same sentence as \"Bruno Wild, Ingenieur\", suggesting a connection between Bruno Wild and Olten. Additionally, the text does not provide any information that would contradict or make this relationship unlikely. The proximity of the names and the lack of contradictory information support the probability of this relationship.",
  "results": [{"at": "PROBABLE"}]}

------------------------

Parsed JSON: [{'at': 'PROBABLE'}]
Extracted Prediction: PROBABLE
❌ Mismatch!

[8] Testing Person: 'Alhert Meyer-Schaffner' | Place: 'Düdingen'
Gold 'at' label: FALSE


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- RAW MODEL OUTPUT ---
{
  "reasoning": "The relationship between Albert Meyer-Schaffner and D\u00fcdingen is probable because the text lists various individuals and their corresponding locations, and Albert Meyer-Schaffner is listed alongside Joseph Vonlanthen, who is also associated with D\u00fcdingen. This proximity in the list suggests a connection between Albert Meyer-Schaffner and D\u00fcdingen. Furthermore, the text does not provide any contradictory information, making the relationship more likely.",
  "results": [{"at": "PROBABLE"}]}

------------------------

Parsed JSON: [{'at': 'PROBABLE'}]
Extracted Prediction: PROBABLE
❌ Mismatch!

[9] Testing Person: 'Max Sauerbrey' | Place: 'La Chaux
de-Fonds'
Gold 'at' label: FALSE


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- RAW MODEL OUTPUT ---
{
  "reasoning": "The relationship between Max Sauerbrey and La Chaux-de-Fonds is probable because Max Sauerbrey is mentioned in the same sentence as Eduard Rutti, who is explicitly stated to be from La Chaux-de-Fonds, suggesting a possible connection or association between the two individuals. Additionally, the text does not provide any explicit information that would contradict or rule out this relationship, making it a plausible inference. The lack of any other information that would suggest Max Sauerbrey is from a different location also supports this probability.",
  "results": [{"at": "PROBABLE"}]}

------------------------

Parsed JSON: [{'at': 'PROBABLE'}]
Extracted Prediction: PROBABLE
❌ Mismatch!

[10] Testing Person: 'Dr. Walter
Deimrelahr, oeiannt,' | Place: 'Küsnacht'
Gold 'at' label: FALSE

--- RAW MODEL OUTPUT ---
{
  "reasoning": "The relationship between Dr. Walter Deimel and Küsnacht is probable because Dr. Walter Deimel is mentioned as being